In [9]:
### Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, cross_val_score
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [10]:
##### Load Data #####
train = pd.read_csv('train_v9rqX0R.csv')
test = pd.read_csv('test_AbJTz2l.csv')

In [11]:
##### Combining two df to ensure consistent transformation  #####
train['source'] = 'train'
test['source'] = 'test'
df = pd.concat([train, test], ignore_index=True)

In [13]:
##### Data Cleaning ######

### Handling Item_Weight missing values using the mean weight of that specific Item_Identifier
item_avg_weight = df.pivot_table(values='Item_Weight', index='Item_Identifier')
def impute_weight(cols):
    weight = cols[0]
    identifier = cols[1]
    if pd.isnull(weight):
        return item_avg_weight.loc[identifier][0] if identifier in item_avg_weight.index else df['Item_Weight'].mean()
    return weight

df['Item_Weight'] = df[['Item_Weight', 'Item_Identifier']].apply(impute_weight, axis=1)

### Handling Outlet_Size missing values using the mode of the specific Outlet_Type
outlet_size_mode = df.pivot_table(values='Outlet_Size', columns='Outlet_Type', aggfunc=(lambda x: x.mode().iat[0]))
def impute_size(cols):
    size = cols[0]
    outlet_type = cols[1]
    if pd.isnull(size):
        return outlet_size_mode.loc['Outlet_Size'][outlet_type]
    return size

df['Outlet_Size'] = df[['Outlet_Size', 'Outlet_Type']].apply(impute_size, axis=1)
df

C:\Users\Umamaheswari.s\AppData\Local\Temp\ipykernel_37196\2655219391.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  weight = cols[0]
C:\Users\Umamaheswari.s\AppData\Local\Temp\ipykernel_37196\2655219391.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  identifier = cols[1]
C:\Users\Umamaheswari.s\AppData\Local\Temp\ipykernel_37196\2655219391.py:16: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  size = cols[0]
C:\Users\Umamahe

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,source
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,train
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,train
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,train
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800,train
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14199,FDB58,10.50,Regular,0.013496,Snack Foods,141.3154,OUT046,1997,Small,Tier 1,Supermarket Type1,NaN,test
14200,FDD47,7.60,Regular,0.142991,Starchy Foods,169.1448,OUT018,2009,Medium,Tier 3,Supermarket Type2,NaN,test
14201,NCO17,10.00,Low Fat,0.073529,Health and Hygiene,118.7440,OUT045,2002,Small,Tier 2,Supermarket Type1,NaN,test
14202,FDJ26,15.30,Regular,0.000000,Canned,214.6218,OUT017,2007,Small,Tier 2,Supermarket Type1,NaN,test


In [14]:
##### Feature Engineering #####

#### Fix Item_Visibility: 0.0 is not impossible, treat as missing and fill with item mean

visibility_avg = df.pivot_table(values='Item_Visibility', index='Item_Identifier')
def impute_visibility(cols):
    vis = cols[0]
    identifier = cols[1]
    if vis == 0:
        return visibility_avg.loc[identifier][0] if identifier in visibility_avg.index else df['Item_Visibility'].mean()
    return vis

df['Item_Visibility'] = df[['Item_Visibility', 'Item_Identifier']].apply(impute_visibility, axis=1)

#### Determine the years of operation (Age of the store)
df['Outlet_Years'] = 2013 - df['Outlet_Establishment_Year']

#### Standardize Item_Fat_Content
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({'LF':'Low Fat', 'reg':'Regular', 'low fat':'Low Fat'})

#### Create a broader category for Item_Type based on Identifier (Food, Drink, Non-Consumable)
df['Item_Combined_Type'] = df['Item_Identifier'].apply(lambda x: x[0:2])
df['Item_Combined_Type'] = df['Item_Combined_Type'].map({'FD':'Food', 'NC':'Non-Consumable', 'DR':'Drinks'})

#### Correct Fat Content for Non-Consumables
df.loc[df['Item_Combined_Type'] == 'Non-Consumable', 'Item_Fat_Content'] = 'Non-Edible'
df

C:\Users\Umamaheswari.s\AppData\Local\Temp\ipykernel_37196\103370448.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  vis = cols[0]
C:\Users\Umamaheswari.s\AppData\Local\Temp\ipykernel_37196\103370448.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  identifier = cols[1]
C:\Users\Umamaheswari.s\AppData\Local\Temp\ipykernel_37196\103370448.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return visibility_avg.loc[identifier][

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,source,Outlet_Years,Item_Combined_Type
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,train,14,Food
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,train,4,Drinks
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,train,14,Food
3,FDX07,19.20,Regular,0.017834,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800,train,15,Food
4,NCD19,8.93,Non-Edible,0.009780,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,train,26,Non-Consumable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14199,FDB58,10.50,Regular,0.013496,Snack Foods,141.3154,OUT046,1997,Small,Tier 1,Supermarket Type1,NaN,test,16,Food
14200,FDD47,7.60,Regular,0.142991,Starchy Foods,169.1448,OUT018,2009,Medium,Tier 3,Supermarket Type2,NaN,test,4,Food
14201,NCO17,10.00,Non-Edible,0.073529,Health and Hygiene,118.7440,OUT045,2002,Small,Tier 2,Supermarket Type1,NaN,test,11,Non-Consumable
14202,FDJ26,15.30,Regular,0.088380,Canned,214.6218,OUT017,2007,Small,Tier 2,Supermarket Type1,NaN,test,6,Food


In [15]:
##### Encoding Categorical Variables  ####

le = LabelEncoder()
df['Outlet'] = le.fit_transform(df['Outlet_Identifier'])
var_mod = ['Item_Fat_Content', 'Outlet_Location_Type', 'Outlet_Size', 'Item_Combined_Type', 'Outlet_Type', 'Outlet']
for i in var_mod:
    df[i] = le.fit_transform(df[i])

#### One-Hot Encoding for the rest
df = pd.get_dummies(df, columns=['Item_Fat_Content', 'Outlet_Location_Type', 'Outlet_Size', 'Item_Combined_Type', 'Outlet_Type', 'Outlet'])
df

,Item_Identifier,Item_Weight,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Item_Outlet_Sales,source,Outlet_Years,...,Outlet_0,Outlet_1,Outlet_2,Outlet_3,Outlet_4,Outlet_5,Outlet_6,Outlet_7,Outlet_8,Outlet_9
0,FDA15,9.30,0.016047,Dairy,249.8092,OUT049,1999,3735.1380,train,14,...,False,False,False,False,False,False,False,False,False,True
1,DRC01,5.92,0.019278,Soft Drinks,48.2692,OUT018,2009,443.4228,train,4,...,False,False,False,True,False,False,False,False,False,False
2,FDN15,17.50,0.016760,Meat,141.6180,OUT049,1999,2097.2700,train,14,...,False,False,False,False,False,False,False,False,False,True
3,FDX07,19.20,0.017834,Fruits and Vegetables,182.0950,OUT010,1998,732.3800,train,15,...,True,False,False,False,False,False,False,False,False,False
4,NCD19,8.93,0.009780,Household,53.8614,OUT013,1987,994.7052,train,26,...,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14199,FDB58,10.50,0.013496,Snack Foods,141.3154,OUT046,1997,NaN,test,16,...,False,False,False,False,False,False,False,False,True,False
14200,FDD47,7.60,0.142991,Starchy Foods,169.1448,OUT018,2009,NaN,test,4,...,False,False,False,True,False,False,False,False,False,False
14201,NCO17,10.00,0.073529,Health and Hygiene,118.7440,OUT045,2002,NaN,test,11,...,False,False,False,False,False,False,False,True,False,False
14202,FDJ26,15.30,0.088380,Canned,214.6218,OUT017,2007,NaN,test,6,...,False,False,True,False,False,False,False,False,False,False


In [18]:
#### Model Building  ####

cols_to_drop = ['Item_Type', 'Outlet_Establishment_Year', 'Outlet_Identifier']
df.drop(columns=cols_to_drop, axis=1, inplace=True, errors='ignore')

#### Split back to train and test
train_df = df.loc[df['source'] == 'train'].copy()
test_df = df.loc[df['source'] == 'test'].copy()

#### Remove the helper columns
train_df.drop('source', axis=1, inplace=True, errors='ignore')
test_df.drop(['source', 'Item_Outlet_Sales'], axis=1, inplace=True, errors='ignore')

#### Define features (X) and target (y)
##### Excluding 'Item_Identifier' from training because it is a label

X = train_df.drop(['Item_Outlet_Sales', 'Item_Identifier'], axis=1)
y = train_df['Item_Outlet_Sales']
X_test_final = test_df.drop(['Item_Identifier'], axis=1)

#### Manual Cross-Validation Loop (To avoid the XGBoost/Sklearn Tag Error)
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_rmse = []

print("Starting Manual Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = XGBRegressor(
        n_estimators=500, 
        learning_rate=0.05, 
        max_depth=5, 
        subsample=0.8, 
        colsample_bytree=0.8,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    val_preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    fold_rmse.append(rmse)
    print(f"Fold {fold} RMSE: {rmse:.4f}")

print(f"\nAverage CV RMSE: {np.mean(fold_rmse):.4f}")

#### Final Training on the full dataset
final_model = XGBRegressor(
    n_estimators=1000, 
    learning_rate=0.05, 
    max_depth=5, 
    subsample=0.8, 
    colsample_bytree=0.8,
    random_state=42
)
final_model.fit(X, y)

print("Final Model Trained Successfully.")

Starting Manual Cross-Validation...
Fold 1 RMSE: 1066.2431
Fold 2 RMSE: 1115.0459
Fold 3 RMSE: 1116.0185
Fold 4 RMSE: 1148.7234
Fold 5 RMSE: 1156.5988

Average CV RMSE: 1120.5259
Final Model Trained Successfully.


In [20]:
#### Create the submission dataframe ####

#### Reset index of test_df to align with the original test file
test_df_reset = test_df.reset_index(drop=True)

#### Re-create the X_test_final with the clean index
X_test_final = test_df_reset.drop(['Item_Identifier'], axis=1)

#### Create the submission dataframe
submission = pd.DataFrame({
    'Item_Identifier': test_df_reset['Item_Identifier'],
    'Outlet_Identifier': test['Outlet_Identifier'],
    'Item_Outlet_Sales': final_model.predict(X_test_final)
})

#### Logic Check: Sales cannot be negative
submission['Item_Outlet_Sales'] = submission['Item_Outlet_Sales'].clip(lower=0)

#### Save to CSV
submission.to_csv('final_submission.csv', index=False)

print("Submission file 'final_submission.csv' created successfully!")
print(f"Submission Shape: {submission.shape}") # Should be (5681, 3)
print(submission.head())

Submission file 'final_submission.csv' created successfully!
Submission Shape: (5681, 3)
  Item_Identifier Outlet_Identifier  Item_Outlet_Sales
0           FDW58            OUT049        1302.868408
1           FDW14            OUT017        1410.926514
2           NCN55            OUT010         803.720215
3           FDQ58            OUT017        3024.017334
4           FDY38            OUT027        6165.859863
